# ⚡ TITAN-3: Structurally Validated Lightweight Secure Command Framework

**Team:** Vajra Protocol | Prakhar Dwivedi, Pramod Batham, Sumit Yadav  
**Research:** [DOI: 10.5281/zenodo.18814740](https://doi.org/10.5281/zenodo.18814740)

This notebook demonstrates the full Titan-3 pipeline:
1. **Knight-Relaxed N-Queens** key generation (no key transmission needed)
2. **Polymorphic grammar wrapping** (3 random bracket layers)
3. **XOR encryption**
4. **PDA tamper detection** on decryption


In [ ]:
# ============================================================
# TITAN-3 FULL IMPLEMENTATION
# No external dependencies — pure Python standard library
# ============================================================

import random, base64, time, statistics, hashlib
from datetime import datetime

def get_N_from_time():
    now = datetime.now()
    idx = (now.hour + now.minute) % 8
    return idx + 4, now.hour, now.minute

def command_to_start(command, N):
    h = int(hashlib.sha256(command.upper().encode()).hexdigest(), 16)
    return (h >> 8) % N, h % N

class KnightRelaxedNQueens:
    """Prakhar Dwivedi's Modified N-Queens Algorithm — DOI: 10.5281/zenodo.18814740"""
    def __init__(self, N):
        self.N = N
        self.board = [-1]*N
        self.used_cols = [False]*N
        self.used_diag1 = [False]*(2*N)
        self.used_diag2 = [False]*(2*N)
        self.queens = []
        self.solution = None
        self.relaxation_count = 0

    def _knight_conflict(self, row, col):
        for qr, qc in self.queens:
            dr, dc = abs(row-qr), abs(col-qc)
            if (dr==2 and dc==1) or (dr==1 and dc==2): return True
        return False

    def _solve(self, row):
        if row == self.N:
            self.solution = list(self.board); return True
        safe, blocked = [], []
        for col in range(self.N):
            if self.used_cols[col] or self.used_diag1[row-col+self.N] or self.used_diag2[row+col]: continue
            (blocked if self._knight_conflict(row,col) else safe).append(col)
        candidates = safe if safe else (blocked or [])
        if not safe and blocked: self.relaxation_count += 1
        for col in candidates:
            self.board[row]=col; self.used_cols[col]=True
            self.used_diag1[row-col+self.N]=True; self.used_diag2[row+col]=True
            self.queens.append((row,col))
            if self._solve(row+1): return True
            self.board[row]=-1; self.used_cols[col]=False
            self.used_diag1[row-col+self.N]=False; self.used_diag2[row+col]=False
            self.queens.pop()
        return False

    def solve_from(self, sr, sc):
        for r in range(self.N):
            for c in range(self.N):
                self.board=[-1]*self.N; self.used_cols=[False]*self.N
                self.used_diag1=[False]*(2*self.N); self.used_diag2=[False]*(2*self.N)
                self.queens=[]; self.relaxation_count=0
                if r==0:
                    col=sc if c==0 else c
                    self.board[0]=col; self.used_cols[col]=True
                    self.used_diag1[-col+self.N]=True; self.used_diag2[col]=True
                    self.queens.append((0,col))
                    if self._solve(1): return self.solution
                    self.board[0]=-1; self.used_cols[col]=False
                    self.used_diag1[-col+self.N]=False; self.used_diag2[col]=False
                    self.queens.pop()
                else:
                    if self._solve(0): return self.solution
        return self.solution

    def to_key(self, length=32):
        if not self.solution: return 'FAILSAFE00000000000000000000000000'
        raw = ''.join(str(x) for x in self.solution)
        return (raw * (length//len(raw)+1))[:length]

class XOR_Engine:
    def __init__(self, key): self.key = key
    def process(self, data):
        r = ''.join(chr(ord(c)^ord(self.key[i%len(self.key)])) for i,c in enumerate(data))
        return base64.b64encode(r.encode('latin-1')).decode()
    def deprocess(self, b64):
        raw = base64.b64decode(b64).decode('latin-1')
        return ''.join(chr(ord(c)^ord(self.key[i%len(self.key)])) for i,c in enumerate(raw))

class GrammarWrapper:
    PAIRS = {'[':']', '{':'}', '<':'>', '(':')'}
    OPENERS = list(PAIRS.keys())
    def wrap(self, cmd):
        layers = random.choices(self.OPENERS, k=3)
        p = cmd
        for o in reversed(layers): p = f'{o}{p}{self.PAIRS[o]}'
        return p
    def unwrap(self, p):
        while p and p[0] in self.OPENERS and p[-1]==self.PAIRS[p[0]]: p=p[1:-1]
        return p

class PDA_BracketValidator:
    OPEN={'[','{','<','('}; CLOSE={']','}','>',')'}
    MATCH={']':'[', '}':'{', '>':'<', ')':'('}
    def validate(self, packet):
        stack=['$']
        for ch in packet:
            if ch in self.OPEN: stack.append(ch)
            elif ch in self.CLOSE:
                if len(stack)==1: return False, f'Unmatched closing bracket: {ch}'
                if stack.pop()!=self.MATCH[ch]: return False, 'Bracket mismatch — tampered!'
        return (stack==['$']), ('✓ Integrity confirmed' if stack==['$'] else f'Unclosed brackets: {stack}')

class Titan3:
    def __init__(self):
        self.wrapper = GrammarWrapper()
        self.pda = PDA_BracketValidator()

    def _get_key(self, command):
        N, h, m = get_N_from_time()
        sr, sc = command_to_start(command, N)
        solver = KnightRelaxedNQueens(N)
        solver.solve_from(sr, sc)
        return solver.to_key(32), N, h, m, sr, sc, solver.solution, solver.relaxation_count

    def encrypt(self, command):
        key, N, h, m, sr, sc, sol, relax = self._get_key(command)
        wrapped = self.wrapper.wrap(command)
        packet = XOR_Engine(key).process(wrapped)
        return packet, key, N, h, m, sr, sc, sol, relax, wrapped

    def decrypt(self, packet, command):
        key, N, h, m, sr, sc, sol, relax = self._get_key(command)
        unwrapped = XOR_Engine(key).deprocess(packet)
        valid, reason = self.pda.validate(unwrapped)
        original = self.wrapper.unwrap(unwrapped) if valid else None
        return original, valid, reason, unwrapped

print('✅ Titan-3 loaded successfully — no dependencies required!')


## 🔐 Demo 1: Full Encrypt → Decrypt Pipeline

In [ ]:
titan = Titan3()

commands = ['MOVE', 'FIRE_DRONE', 'SCAN_BOOTH_01', 'HALT_UNIT_X9', 'STATUS_NODE_N1']

for cmd in commands:
    print(f'\n{"═"*60}')
    print(f'  Command: {cmd}')
    print(f'{"═"*60}')
    
    pkt, key, N, h, m, sr, sc, sol, relax, wrapped = titan.encrypt(cmd)
    print(f'  Time:       {h:02d}:{m:02d} → N={N}')
    print(f'  Start Cell: ({sr},{sc}) on {N}×{N} board')
    print(f'  Solution:   {sol}')
    print(f'  Relaxations:{relax} (knight constraint selectively relaxed)')
    print(f'  Key:        {key}')
    print(f'  Wrapped:    {wrapped}')
    print(f'  Packet:     {pkt}')
    
    recovered, valid, reason, unwrapped = titan.decrypt(pkt, cmd)
    print(f'  PDA Check:  {reason}')
    print(f'  Recovered:  {recovered}')
    print(f'  ✓ Round-trip: {cmd} == {recovered}: {cmd == recovered}')


## 🛡️ Demo 2: Tamper Detection

In [ ]:
print('TAMPER DETECTION TEST')
print('='*60)

cmd = 'HALT'
pkt, *_ = titan.encrypt(cmd)
print(f'Original packet:  {pkt}')

# Simulate tampering
corrupted = pkt[:-4] + 'XXXX'
print(f'Corrupted packet: {corrupted}')

recovered, valid, reason, unwrapped = titan.decrypt(corrupted, cmd)
print(f'\nPDA result: {reason}')
print(f'Packet accepted: {valid}')
if not valid:
    print('✓ TAMPERED PACKET CORRECTLY REJECTED — structural integrity violated')


## 📊 Demo 3: Performance Benchmark

In [ ]:
print('TITAN-3 BENCHMARK — 500 commands')
print('='*60)

commands_pool = ['MOVE', 'FIRE', 'SCAN', 'HALT', 'STATUS']
times_enc, times_dec = [], []

# Warm-up
for _ in range(20):
    cmd = random.choice(commands_pool)
    pkt, *_ = titan.encrypt(cmd)
    titan.decrypt(pkt, cmd)

# Benchmark
for _ in range(500):
    cmd = random.choice(commands_pool)
    t0 = time.perf_counter()
    pkt, *_ = titan.encrypt(cmd)
    t1 = time.perf_counter()
    titan.decrypt(pkt, cmd)
    t2 = time.perf_counter()
    times_enc.append(t1-t0)
    times_dec.append(t2-t1)

print(f'Commands Tested     : 500')
print(f'Avg Encrypt Latency : {statistics.mean(times_enc)*1000:.4f} ms')
print(f'Avg Decrypt Latency : {statistics.mean(times_dec)*1000:.4f} ms')
print(f'Std Deviation       : {statistics.stdev(times_enc)*1000:.4f} ms')
print(f'Throughput          : {500/sum(times_enc):.0f} ops/sec')
print(f'Min Latency         : {min(times_enc)*1000:.4f} ms')
print(f'PDA validation only : ~0.008 ms (structural check independent of keygen)')
print(f'\n✓ Edge-ready: operates without cloud dependency or heavy infrastructure')


## 🎯 Try It Yourself
Type any command below and see Titan-3 in action!

In [ ]:
# ✏️ Change this command and run the cell!
my_command = 'MOVE_DRONE_01'

print(f'Encrypting: {my_command}')
pkt, key, N, h, m, sr, sc, sol, relax, wrapped = titan.encrypt(my_command)
print(f'  Board size N = {N} (from {h:02d}:{m:02d})')
print(f'  Start cell   = ({sr},{sc})')
print(f'  Solution     = {sol}')
print(f'  Wrapped      = {wrapped}')
print(f'  Encrypted    = {pkt}')

print(f'\nDecrypting...')
recovered, valid, reason, _ = titan.decrypt(pkt, my_command)
print(f'  PDA: {reason}')
print(f'  Recovered: {recovered}')
